# Этап 2. Feature Engineering

**Цель:** превратить «сырые» транзакции в матрицу признаков, в которой отмывание
отличается от обычного поведения, — и не наделать при этом утечек.

## Главная идея этапа: признак = численное выражение типологии

| Типология отмывания | Что происходит | Какой признак это ловит |
|---|---|---|
| `Structuring` | дробят сумму чуть ниже порога $10 000 | `amount_near_threshold`, `vel_near_threshold_count_24h` |
| `Smurfing` | серия мелких платежей за короткое время | `vel_count_1h`, `vel_count_24h`, inter-arrival |
| `Fan_Out` / `Fan_In` | один → много / много → один | `g_out_degree`, `g_in_degree`, `sender_n_receivers` |
| `Cycle` | деньги ходят по кругу | `g_in_cycle`, `g_in_repeat_cycle`, `g_is_mutual_pair` |
| `Bipartite` / `Gather-Scatter` | группа счетов, «все со всеми» | `g_component_size`, `g_core_component_size`, `g_pagerank` |
| `Single_large`, `Over-Invoicing` | одна огромная сумма | `amount_log`, `amount_pct_in_currency`, `amount_to_sender_median` |
| `Behavioural_Change_1/2` | резкая смена поведения | `amount_zscore_sender`, `vel_amount_24h_to_median` |
| `Deposit-Send`, `Cash_Withdrawal` | наличные на входе | `payment_type_code` (Cash Deposit / Withdrawal) |

## Три правила, без которых всё остальное не имеет смысла

1. **Никакого будущего.** Профили счетов, пары и граф считаются ТОЛЬКО на train.
2. **Окна — назад.** Velocity считает операции, которые УЖЕ случились к моменту транзакции.
3. **Каждый признак проверяем на lift.** Признак без lift > 1 — это шум, в модель его не берём.

План тетрадки:
1. Разбиение по времени (train / val / test) + проверка на утечку
2. Потоковая сборка полной матрицы на ВСЕХ 9.5 млн строк
3. Рабочая выборка для демонстраций
4. Транзакционные признаки
5. Поведенческие профили (FIT -> APPLY) и демонстрация утечки «на живом примере»
6. Velocity: скользящие окна + сверка векторного расчёта с брутфорсом
7. Парные признаки
8. Граф-признаки: степени, PageRank, циклы, компоненты
9. Контроль качества: утечки, пропуски, константы, lift-рейтинг


In [ ]:
# ============================================================
# 0. НАСТРОЙКА
# ============================================================
import sys, warnings, gc
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src" / "data_loader.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import (
    load_dataset, memory_mb, COL_AMOUNT, COL_SENDER, COL_RECEIVER,
    COL_PAY_TYPE, COL_TARGET, COL_LAUND_TYPE, STRUCTURING_THRESHOLD,
    FIGURES_DIR, FEATURES_DIR, PROCESSED_DIR,
)
from src.eda_utils import rate_by_bucket, bucketize, two_proportion_ztest, save_fig
from src.split import time_split, assert_no_leakage, check_feature_leakage
from src.features import (
    add_transaction_features, fit_amount_percentiles, add_amount_percentile,
    fit_account_profiles, add_behavioural_features,
    compute_velocity_features, add_velocity_features,
    fit_pair_stats, add_pair_features,
    build_and_save_features, FEATURE_COLUMNS, GRAPH_FEATURE_COLUMNS,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
print("pandas:", pd.__version__)

In [ ]:
# ============================================================
# 1. РАЗБИЕНИЕ ПО ВРЕМЕНИ (train / val / test)
# ============================================================
# Полные 9.5 млн строк в эту тетрадку мы НЕ грузим: они нужны только один раз —
# чтобы собрать матрицу признаков, и это делает отдельный процесс (следующая
# ячейка). Здесь же берём рабочую выборку ~1.9 млн строк: читаем два row group
# подготовленного parquet-кэша (файл перемешан, так что это честная выборка).
import pyarrow.parquet as pq
from src.data_loader import cache_path

_PREP = pq.ParquetFile(cache_path())
_ng = _PREP.metadata.num_row_groups
_take = [_ng // 3, 2 * _ng // 3]                  # два среза из разных мест файла
work = pd.concat([_PREP.read_row_group(i).to_pandas() for i in _take], ignore_index=True)
del _PREP
gc.collect()
work = work.sort_values("txn_ts").reset_index(drop=True)   # порядок важен для velocity

print(f"Рабочая выборка: {len(work):,} строк (row groups {_take} из {_ng}) "
      f"| память {memory_mb(work):,.0f} MB")
print(f"Base rate: {work[COL_TARGET].mean() * 100:.4f}% "
      f"| positives: {int(work[COL_TARGET].sum()):,}")

train_mask, val_mask, test_mask = time_split(work, train_frac=0.70, val_frac=0.15)

# Метка периода: удобно резать train/val/test, не пересчитывая границы заново.
period_w = pd.Series("train", index=work.index, dtype=object)
period_w[val_mask] = "val"
period_w[test_mask] = "test"
print("\nРазбиение по периодам:")
print(period_w.value_counts().to_frame("строк"))

assert_no_leakage(work, train_mask, test_mask)


### Почему разбиение по времени — это не formalность

Представь, что ты перемешал данные случайно. Тогда:

* в train попадёт транзакция от 1 декабря, а в test — от 15 ноября;
* признак «средняя сумма счёта» посчитается с учётом декабрьской операции;
* модель выучит: «у этого счёта в среднем большие суммы → он подозрителен»,
  опираясь на то, что случится в будущем;
* PR-AUC на test будет красивым, а в боевом режиме (где будущего нет) — нет.

`assert_no_leakage` выше — дешёвая страховка: если границы нарушены, она упадёт.

---
# 2. Потоковая сборка полной матрицы

Матрица 9.5 млн строк × 60 признаков во float32 — это ~2.3 ГБ. Держать её в памяти
(да ещё вместе с исходным датафреймом) непозволительно. Поэтому:

* velocity уже посчитана один раз на всех данных (ей нужна история);
* профили / пары / граф — маленькие таблицы, они в памяти целиком;
* всё остальное собирается **чанками**, и каждый чанк сразу дописывается в parquet.

Такой же приём применяют в продакшене, где счёт идёт на сотни миллионов транзакций.

In [ ]:
# ============================================================
# 2. ПОТОКОВАЯ СБОРКА ПОЛНОЙ МАТРИЦЫ (все 9.5 млн строк)
# ============================================================
# Тяжёлую сборку запускаем ОТДЕЛЬНЫМ ПРОЦЕССОМ — скриптом src/build_features.py.
#
# Почему именно так, а не «просто выполнить код в этой тетрадке»:
#   * скрипт читает полные данные, сам считает разбиение по времени и собирает
#     матрицу чанками — это почти 3 ГБ памяти на пике;
#   * у тетрадки своя память уже занята (выборка, графики), лимит ОЗУ общий,
#     и вместе они бы просто не влезли;
#   * ровно так же это работает в продакшене: ночная джоба считает признаки
#     для свежих транзакций, а аналитик смотрит результат.
import subprocess


def rss_mb() -> float:
    """Сколько памяти процесс занимает ПРЯМО СЕЙЧА (не накопленный максимум)."""
    return int(open("/proc/self/statm").read().split()[1]) * 4096 / 1e6


print(f"Память тетрадки перед сборкой: {rss_mb():,.0f} MB")
cmd = [sys.executable, "-m", "src.build_features",
       "--chunk-size", "500000", "--max-edges", "200000"]
print("Запускаю:", " ".join(cmd), "\n")
res = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("\n".join(res.stdout.strip().splitlines()[-14:]))
if res.returncode != 0:
    raise RuntimeError(f"Сборка признаков упала с кодом {res.returncode}")

---
# 4. Транзакционные признаки

Признаки, которые видны из самой строки: сумма, порог, время, канал, география.
Утечки在这里 нет по определению — мы ничего не смотрим «вокруг».

In [ ]:
# ============================================================
# 4. ТРАНЗАКЦИОННЫЕ ПРИЗНАКИ
# ============================================================
work = add_transaction_features(work)
grid = fit_amount_percentiles(work[train_mask])       # линейка процентилей — тоже с train
work = add_amount_percentile(work, grid)
print(f"Добавлены признаки. Память work: {memory_mb(work):,.0f} MB")

tx_cols = ["amount_log", "amount_pct_in_currency", "amount_near_threshold",
           "amount_above_threshold", "is_round_100", "is_cross_border",
           "is_currency_mismatch", "is_night", "is_weekend"]
display(work[tx_cols].describe().T)

In [ ]:
# ---- Lift по транзакционным признакам -------------------------------
# Метрика простая: режем признак на 5 корзин по квантилям и смотрим,
# в какой корзине доля отмывания максимальна относительно base rate.
base_rate = work[COL_TARGET].mean()

def lift_table(X: pd.DataFrame, y: pd.Series, cols: list[str], n_bins: int = 5,
               min_n: int = 200) -> pd.DataFrame:
    rows = []
    for c in cols:
        s = X[c]
        if s.nunique(dropna=True) < n_bins:
            continue
        try:
            q = np.nanquantile(s.dropna().to_numpy(), np.linspace(0, 1, n_bins + 1))
            q = np.unique(q)
            b = pd.cut(s, bins=q, include_lowest=True)
            t = y.groupby(b, observed=True).agg(n="size", pos="sum")
            t = t[t["n"] >= min_n]
            if len(t) == 0:
                continue
            rate = t["pos"] / t["n"]
            rows.append({"признак": c, "max_lift": rate.max() / base_rate,
                         "min_lift": rate.min() / base_rate,
                         "корзина_макс": str(rate.idxmax())})
        except Exception as e:
            rows.append({"признак": c, "max_lift": np.nan, "min_lift": np.nan,
                         "корзина_макс": f"ошибка: {e}"})
    return pd.DataFrame(rows).sort_values("max_lift", ascending=False).reset_index(drop=True)

lift_tx = lift_table(work[tx_cols], work[COL_TARGET], tx_cols)
display(lift_tx)

---
# 5. Поведенческие профили: FIT на train → APPLY ко всем

**Идея behavioural profiling:** подозрительна не сумма сама по себе, а сумма,
***непохожая на обычную для этого счёта***. Счёт, который год платил по $500 и
вдруг отправил $20 000, — это нетипично, даже если $20 000 для банка не деньги.

Профиль считаем по train и «приклеиваем» к val/test **без пересчёта**.

In [ ]:
# ============================================================
# 5. ПОВЕДЕНЧЕСКИЕ ПРОФИЛИ
# ============================================================
prof = fit_account_profiles(work[train_mask])
print(f"Профилей счетов: {len(prof):,}")
display(prof.describe().T[["mean", "50%", "max"]])

work = add_behavioural_features(work, prof)
beh_cols = ["amount_to_sender_median", "amount_to_receiver_median",
            "amount_zscore_sender", "amount_to_sender_max",
            "sender_txn_count", "sender_n_receivers", "sender_txn_per_active_day",
            "receiver_txn_count", "receiver_n_senders"]
display(lift_table(work[beh_cols], work[COL_TARGET], beh_cols))
gc.collect()

## 5.1 Демонстрация утечки «на живом примере»

Считаем один и тот же профиль двумя способами и сравниваем:

* **ПРАВИЛЬНО** — медиана суммы счёта по train-периоду;
* **НЕПРАВИЛЬНО** — медиана по ВСЕМУ датасету (включая будущее).

Разница выглядит небольшой, но именно она «подсвечивает» модели счета, которые
станут подозрительными в будущем. Ниже — численная оценка масштаба подглядывания.

In [ ]:
# ---- Сколько информации о будущем подсматривает «неправильный» профиль ----
prof_all = fit_account_profiles(work)                     # НЕПРАВИЛЬНО: всё данные сразу
cmp = prof[["sender_median_amount"]].join(
    prof_all[["sender_median_amount"]], how="inner", rsuffix="_all")
cmp = cmp.dropna()
diff = (cmp["sender_median_amount"] - cmp["sender_median_amount_all"]).abs()

print(f"Счетов в сравнении: {len(cmp):,}")
print(f"Медиана профиля отличается более чем на 1%: {(diff / cmp['sender_median_amount'] > 0.01).mean() * 100:.1f}% счетов")
print(f"Максимальное отличие: {(diff / cmp['sender_median_amount']).max() * 100:.1f}%")

# Главный эффект: для счетов, у которых в test-периоде активность МЕНЯЕТСЯ,
# «вседанный» профиль уже знает будущее. Проверим на транзакциях из test:
test_senders = set(work.loc[test_mask, COL_SENDER].unique())
cmp_test = cmp[cmp.index.isin(test_senders)]
print(f"\nСреди счетов, работающих в test-периоде ({len(cmp_test):,}):")
print(f"  профиль отличается >10%: "
      f"{((cmp_test['sender_median_amount'] - cmp_test['sender_median_amount_all']).abs() / cmp_test['sender_median_amount'] > 0.10).mean() * 100:.1f}%")

print("\nВЫВОД: разница есть у заметной доли счетов. В train она выглядит как")
print("«полезный сигнал», а в продакшене его просто не будет — модель потеряет")
print("качество именно на новых данных.")
del prof_all, cmp, cmp_test
gc.collect()

---
# 6. Velocity: признаки скользящего окна

**Бизнес-смысл:** smurfing/structuring — это всегда **серия** операций.
Одна транзакция на $9 500 может быть случайностью; пять таких за сутки — схема.

Считаем для каждого счёта за окна 1ч / 6ч / 24ч / 7д:
`vel_count_*` (сколько операций, включая текущую), `vel_amount_sum_*` (оборот),
`vel_near_threshold_count_*` (сколько операций прижато к порогу $10 000).

In [ ]:
# ============================================================
# 6.1 ПРОВЕРКА АЛГОРИТМА: векторный расчёт vs брутфорс
# ============================================================
# Окна по 9.5 млн строк считаются хитрым приёмом (пара «счёт + время» упаковывается
# в одно число int64, дальше работает обычный searchsorted). Прежде чем верить
# такому коду, проверим его на маленькой выборке прямым перебором.

from src.features import _rolling_window_stats

test_small = work.head(5_000)
codes = test_small[COL_SENDER].cat.codes.to_numpy()
codes = np.where(codes < 0, codes.max() + 1, codes).astype(np.int64)
ts = test_small["txn_ts"].to_numpy().astype("datetime64[s]").astype(np.int64)
amt = test_small[COL_AMOUNT].to_numpy(dtype="float64")

stats = _rolling_window_stats(codes, ts, {"amount": amt}, (24,))
vec_count, vec_sum = stats["_count_24"], stats["_sum_amount_24"]

brute_count = np.zeros(len(test_small), dtype=np.int64)
brute_sum = np.zeros(len(test_small), dtype=np.float64)
for i in range(len(test_small)):
    m = (codes == codes[i]) & (ts > ts[i] - 24 * 3600) & (ts <= ts[i])
    brute_count[i] = m.sum()
    brute_sum[i] = amt[m].sum()

print(f"count совпадает: {np.array_equal(vec_count, brute_count)}")
print(f"sum   совпадает: {np.allclose(vec_sum, brute_sum)}")
print("Если оба True — векторныйvelocity считает ровно то же, что и лобовой перебор,")
print("но за секунды вместо часов.")

In [ ]:
# ============================================================
# 6.2 СЧИТАЕМ VELOCITY НА ВСЕХ ДАННЫХ
# ============================================================
# Velocity считаем ОДИН РАЗ на всём датасете: окнам нужна полная история счёта,
# и если резать данные на части, на границах частей окна «обрубятся».
vel = compute_velocity_features(work)
print(f"Velocity признаков: {vel.shape[1]} | память: {memory_mb(vel):,.0f} MB")
display(vel.describe().T[["mean", "50%", "max"]])

# Lift считаем на временной копии (полный merge делать не нужно — дорого по памяти)
tmp = pd.concat([work[COL_TARGET].reset_index(drop=True), vel.reset_index(drop=True)], axis=1)
display(lift_table(tmp, tmp[COL_TARGET], list(vel.columns)))
del tmp
gc.collect()

In [ ]:
# ---- Визуализация: риск vs частота операций и интервал между ними ----
vel_plot = vel.copy()
vel_plot[COL_TARGET] = work[COL_TARGET].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

b1 = bucketize(vel_plot["vel_count_24h"], [1, 2, 3, 4, 6, 10, 20, np.inf],
               labels=["1", "2", "3", "4-5", "6-9", "10-19", "20+"])
t1 = rate_by_bucket(vel_plot.assign(_b=b1), "_b")
axes[0].bar(t1.index.astype(str), t1["lift"], color="#C44E52")
axes[0].axhline(1.0, color="grey", lw=1)
axes[0].set_title("Lift vs число операций счёта за 24 часа")
axes[0].set_xlabel("операций за сутки"); axes[0].set_ylabel("lift")

b2 = bucketize(vel_plot["vel_amount_sum_24h"], [0, 1e3, 5e3, 1e4, 2e4, 5e4, 1e5, np.inf],
               labels=["<1k", "1-5k", "5-10k", "10-20k", "20-50k", "50-100k", ">100k"])
t2 = rate_by_bucket(vel_plot.assign(_b=b2), "_b")
axes[1].bar(t2.index.astype(str), t2["lift"], color="#4C72B0")
axes[1].axhline(1.0, color="grey", lw=1)
axes[1].set_title("Lift vs оборот счёта за 24 часа")
axes[1].set_xlabel("оборот за сутки"); axes[1].set_ylabel("lift")
axes[1].tick_params(axis="x", rotation=30)
save_fig("18_velocity_lift", fig)
plt.show()
del vel_plot
gc.collect()

---
# 7. Парные признаки

**Бизнес-смысл:** «накатанная дорожка» между двумя счетами (переводят друг другу
регулярно) — это совсем другое, чем разовый перевод новому контрагенту.
Обе ситуации бывают подозрительны, но по-разному:

* новая пара + крупная сумма → классический сценарий «money mule»;
* пара с 50 переводами за день → structuring/fan-out.

In [ ]:
# ============================================================
# 7. ПАРНЫЕ ПРИЗНАКИ
# ============================================================
pair = fit_pair_stats(work[train_mask])
print(f"Пар в train: {len(pair):,}")
display(pair[["pair_count", "pair_amount_sum"]].describe().T)

work = add_pair_features(work, pair)
pair_cols = ["pair_count", "pair_amount_sum", "pair_amount_mean", "is_new_pair"]
display(lift_table(work[pair_cols], work[COL_TARGET], pair_cols))
gc.collect()

---
# 8. Граф-признаки

Здесь отмывание видно как **структура**, а не как отдельная операция.

Считаем:
* **степени и обороты** (`g_out_degree`, `g_in_degree`, `g_out_amount_sum`) — fan-out/fan-in;
* **PageRank** — «важность» счёта в сети переводов (считаем сами на разреженных
  матрицах: networkx на 7 млн рёбер не влезает в память);
* **участие в цикле** — через «отшелушивание» вершин с нулевой степенью
  (в ациклическом графе всегда есть вершина без входа и вершина без выхода,
  значит после удаления таких вершин остаются ровно те, что лежат на циклах);
* **размер компоненты связности** — размер «компании» счетов вокруг счёта;
* **взаимные пары** (A→B и B→A) — цикл длины 2.

In [ ]:
# ============================================================
# 8. ГРАФ-ПРИЗНАКИ (FIT на train)
# ============================================================
from src.graph_features import (
    build_edge_list, compute_degrees, sparse_pagerank, cyclic_core,
    component_sizes, build_filtered_graph, cycle_and_component_features,
    fit_graph_tables, merge_graph_tables,
)

edges = build_edge_list(work[train_mask])
print(f"Рёбер в графе train: {len(edges):,}")
display(edges.head())

deg = compute_degrees(edges)
pr = sparse_pagerank(edges, verbose=True)
print(f"PageRank посчитан для {len(pr):,} счетов")

deg = deg.join(pr, how="outer")
display(deg.describe().T[["mean", "50%", "max"]])

In [ ]:
# ---- Циклы: «отшелушивание» вершин с нулевой степенью -----------------
core_nodes, core_edges = cyclic_core(edges, verbose=True)
print(f"\nСчетов, лежащих на циклах: {len(core_nodes):,} "
      f"({len(core_nodes) / len(deg) * 100:.1f}% от графа)")

# Граф ПОВТОРЯЮЩИХСЯ связей: он намного разреженнее, поэтому избирательнее.
rep = edges[edges["n_txn"] >= 2]
print(f"Повторяющихся связей (n_txn >= 2): {len(rep):,}")
if len(rep) > 0:
    rep_core, _ = cyclic_core(rep)
    print(f"Счетов в циклах по повторяющимся связям: {len(rep_core):,}")
else:
    rep_core = pd.Index([])

In [ ]:
# ---- Проверка алгоритма на игрушечном графе --------------------------
# Прежде чем верить результату «100% счетов в циклах», проверим, что сам
# алгоритм вообще умеет находить циклы. Берём граф, который весь помещается
# в голове:
#     A -> B -> C -> A        (цикл длины 3)
#     D <-> E                 (цикл длины 2)
#     F -> A                  (ведёт В цикл, но сам «тупиковый»: входящих нет)
tiny = pd.DataFrame({
    "Sender_account":   ["A", "B", "C", "D", "E", "F"],
    "Receiver_account": ["B", "C", "A", "E", "D", "A"],
})
tiny_core, _ = cyclic_core(tiny, verbose=False)
print("Вершины, лежащие на циклах:", sorted(tiny_core))
print("Ожидаем ['A', 'B', 'C', 'D', 'E']: F отшелушится первым — у него нет")
print("входящих рёбер, а значит на цикле он лежать не может.")

**Честно про результат «100 % счетов в циклах».**

Алгоритм работает (только что проверили на игрушечном графе), но в нашем
синтетическом датасете он почти ничего не разделяет: у генератора получатели
выбираются в основном случайно, поэтому граф переводов получается «плотным» —
средняя степень счёта около 20, и между любыми двумя счетами почти наверняка
есть путь в обе стороны. В таком графе цикл найдётся у всех.

Что с этим делать:

* **в проде** признак «лежит на цикле» обычно избирателен: реальные сети
  переводов устроены сообществами, и средняя степень там на порядок ниже;
* **здесь** `g_in_cycle` окажется константой — мы это увидим в контроле качества
  и выбросим его из модели;
* **избирательная версия** — `g_in_repeat_cycle`: циклы только по связям, по
  которым прошло хотя бы 2 платежа. Такой граф разреженнее, и в него попадает
  уже не 100 % счетов, а заметно меньше.

In [ ]:
# ---- Визуализация: маленький фрагмент сети с циклом -------------------
# Строим подграф вокруг нескольких счетов из циклического ядра, чтобы увидеть
# структуру глазами (на всём графе 100+ тыс. вершин — рисунок был бы кашей).
import networkx as nx

if len(core_nodes) > 0:
    sub_edges = core_edges.head(200)
    start = sub_edges[COL_SENDER].iloc[0]
    G_small = nx.DiGraph()
    for u, v, w in zip(sub_edges[COL_SENDER], sub_edges[COL_RECEIVER], sub_edges["n_txn"]):
        G_small.add_edge(u, v, weight=float(w))
    # берём окрестность стартовой вершины, чтобы получить связный фрагмент
    nodes_around = set([start]) | set(G_small.successors(start)) | set(G_small.predecessors(start))
    for n in list(nodes_around):
        if len(nodes_around) > 30:
            break
        nodes_around |= set(G_small.successors(n)) | set(G_small.predecessors(n))
    G_draw = G_small.subgraph(list(nodes_around)[:40]).copy()

    fig, ax = plt.subplots(figsize=(10, 8))
    pos = nx.spring_layout(G_draw, seed=RANDOM_STATE)
    nx.draw_networkx_nodes(G_draw, pos, ax=ax, node_size=300, node_color="#C44E52", alpha=.85)
    nx.draw_networkx_edges(G_draw, pos, ax=ax, arrows=True, arrowsize=12,
                           edge_color="grey", alpha=.7)
    nx.draw_networkx_labels(G_draw, pos, ax=ax, font_size=6)
    ax.set_title("Фрагмент циклического ядра графа переводов\n"
                 "(вершины, лежащие на циклах: деньги могут вернуться к отправителю)")
    ax.axis("off")
    save_fig("19_graph_cycle_fragment", fig)
    plt.show()
else:
    print("Циклическое ядро пусто — визуализировать нечего.")

In [ ]:
# ---- Приклеиваем граф-признаки ко всем транзакциям --------------------
accounts, mutual, gmeta = fit_graph_tables(work[train_mask], max_edges=400_000, verbose=True)
work = merge_graph_tables(work, accounts, mutual)

g_cols = [c for c in GRAPH_FEATURE_COLUMNS if c in work.columns]
print(f"\nГраф-признаков: {len(g_cols)}")
display(lift_table(work[g_cols], work[COL_TARGET], g_cols))
gc.collect()

---
# 9. Контроль качества признаков

Проверяем четыре вещи. Каждая из них хотя бы раз в жизни ломала реальный проект:

1. **Утечка** — нет ли признака, который один «предсказывает» таргет (|corr| > 0.9)?
2. **Пропуски** — откуда NaN и не слишком ли их много?
3. **Нулевая дисперсия** — признак-константа не несёт информации.
4. **Lift** — есть ли у признака вообще сигнал?

In [ ]:
# ============================================================
# 9. ПРОВЕРКИ (на выборке из parquet, чтобы не грузить 2.3 ГБ)
# ============================================================
import pyarrow.parquet as pq

PF_PATH = FEATURES_DIR / "features_full.parquet"
pf = pq.ParquetFile(PF_PATH)
print(f"Parquet: {pf.metadata.num_rows:,} строк, {pf.metadata.num_row_groups} row groups")

# Читаем не больше двух row groups целиком: этого хватает для всех проверок
# (2 млн строк), а память остаётся в пределах разумного.
rg = pf.metadata.num_row_groups
take = list(range(0, rg, max(1, rg // 2)))[:2]
print(f"Читаем row groups: {take} из {rg}")
sample_parts = [pf.read_row_group(i).to_pandas() for i in take]
fs = pd.concat(sample_parts, ignore_index=True)
del sample_parts
gc.collect()
print(f"Выборка для проверок: {len(fs):,} строк | память {memory_mb(fs):,.0f} MB")

feat_cols = [c for c in fs.columns if c not in (COL_TARGET, "period")]
y_s = fs[COL_TARGET].astype("int8")
print(f"Признаков: {len(feat_cols)} | base rate в выборке: {y_s.mean() * 100:.4f}%")

check_feature_leakage(fs[feat_cols], y_s, threshold=0.90)

In [ ]:
# ---- Пропуски и константы -------------------------------------------
miss = (fs[feat_cols].isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
print("Признаки с пропусками (% строк):")
display(miss.round(3).to_frame("пропуски_%") if len(miss) else "пропусков нет")

nuniq = fs[feat_cols].nunique()
const_cols = list(nuniq[nuniq <= 1].index)
print(f"\nКонстантных признаков: {len(const_cols)} {const_cols if const_cols else ''}")
print("(Константы нужно выбросить: модель не сможет по ним ничего выучить.)")

In [ ]:
# ---- Итоговый lift-рейтинг признаков --------------------------------
feat_lift = lift_table(fs[feat_cols], y_s, feat_cols, n_bins=5, min_n=500)
display(feat_lift.head(25))

fig, ax = plt.subplots(figsize=(10, 9))
top = feat_lift.dropna(subset=["max_lift"]).head(25).iloc[::-1]
colors = ["#C44E52" if v >= 2 else "#DD8452" if v >= 1.5 else "#4C72B0" for v in top["max_lift"]]
ax.barh(top["признак"], top["max_lift"], color=colors)
ax.axvline(1.0, color="grey", lw=1)
ax.axvline(2.0, color="#C44E52", ls="--", lw=1, label="lift = 2")
ax.set_title("Топ-25 признаков по lift (насколько «грязнее» лучшая корзина)")
ax.set_xlabel("max lift к base rate")
ax.legend()
save_fig("20_feature_lift_ranking", fig)
plt.show()

In [ ]:
# ---- Корреляции признаков (независимость важна для интерпретации) ----
num_feats = [c for c in feat_cols if fs[c].nunique() > 3]
corr_m = fs[num_feats].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(corr_m, cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": .7},
            xticklabels=True, yticklabels=True, ax=ax)
ax.set_title("Корреляции признаков (красное = дублируют друг друга)")
ax.tick_params(labelsize=6)
save_fig("21_feature_correlations", fig)
plt.show()

# Сильно скоррелированные пары: кандидаты на удаление
hi = []
cols = corr_m.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        v = corr_m.iloc[i, j]
        if pd.notna(v) and abs(v) > 0.9:
            hi.append((cols[i], cols[j], round(float(v), 3)))
print(f"Пар с |corr| > 0.9: {len(hi)}")
for a, b, v in hi[:20]:
    print(f"  {a} ~ {b}: {v}")

In [ ]:
# ============================================================
# 10. ИТОГИ ЭТАПА 2
# ============================================================
summary = {
    "строк_в_матрице": int(pf.metadata.num_rows),
    "признаков": len(feat_cols),
    "граф_рёбер_train": gmeta["n_edges_train"],
    "граф_счетов": gmeta["n_accounts"],
    "счетов_в_циклах": gmeta["n_accounts_in_cycle"],
    "счетов_в_циклах_повтор": gmeta["n_accounts_in_repeat_cycle"],
    "взаимных_пар": gmeta["n_mutual_pairs"],
    "признаков_с_lift_2+": int((feat_lift["max_lift"] >= 2).sum()) if "max_lift" in feat_lift else 0,
    "константных_признаков": len(const_cols),
    "макс_corr_с_таргетом": round(float(
        max(abs(np.corrcoef(fs[c].fillna(0), y_s)[0, 1]) for c in feat_cols)), 4),
}
for k, v in summary.items():
    print(f"{k:26s}: {v}")

print("ПОЯСНЕНИЕ К КОНСТАНТАМ: граф-признаки g_in_cycle / g_component_size /")
print("g_core_component_size / g_receiver_in_cycle выродились в константы — в синтетическом")
print("графе получатели выбираются почти случайно, поэтому он плотный и цикл есть у всех.")
print("Код корректен (проверен на игрушечном графе), на реальных данных метрика избирательна.\n")

checklist = [
    "[x] Разбиение по времени train/val/test + проверка на утечку",
    "[x] Транзакционные признаки (сумма, порог, время, канал, география)",
    "[x] Поведенческие профили: FIT на train -> APPLY ко всем периодам",
    "[x] Демонстрация, чем опасен профиль, посчитанный по всем данным",
    "[x] Velocity на скользящих окнах + сверка векторного расчёта с брутфорсом",
    "[x] Парные признаки (новая пара / накатанная дорожка)",
    "[x] Граф-признаки: степени, PageRank, циклы, компоненты, взаимные пары",
    "[x] Потоковая сборка 9.5 млн строк в parquet (чанками)",
    "[x] Контроль качества: утечки, пропуски, константы, lift-рейтинг",
    "[x] data/features/features_full.parquet готов к Этапу 3-4",
]
print("\n".join(checklist))
print("\nЭТАП 2 ГОТОВ. Дальше: Этап 3 — Rule-based baseline (красные флаги).")